# 🤖 Notebook 03 — Model Training, Hyperparameter Tuning & Evaluation

**Smart Home Energy Consumption Prediction**

> Objective: Train multiple ML models, tune the best ones, evaluate performance, and export the final model.

---

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, warnings, os, time
warnings.filterwarnings('ignore')
os.makedirs('../outputs', exist_ok=True)

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_train = pd.read_csv('../data/X_train.csv')
X_test  = pd.read_csv('../data/X_test.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/y_test.csv').squeeze()

print("Train:", X_train.shape, "| Test:", X_test.shape)

def evaluate(name, y_true, y_pred, t):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    return {'Model': name, 'MAE': round(mae,3), 'RMSE': round(rmse,3),
            'R2': round(r2,4), 'Train_Time_s': round(t,3)}

FileNotFoundError: [Errno 2] No such file or directory: '../data/X_train.csv'

## 1. Baseline Models

In [ ]:
results = []
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression':  Ridge(alpha=1.0),
    'Lasso Regression':  Lasso(alpha=0.1),
    'Decision Tree':     DecisionTreeRegressor(random_state=42),
}

for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    t = time.time() - t0
    pred = model.predict(X_test)
    results.append(evaluate(name, y_test, pred, t))
    print(f"{name:25s} | R²={results[-1]['R2']:.4f} | MAE={results[-1]['MAE']:.3f} | time={t:.3f}s")

## 2. Random Forest — Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 150, 200],
    'max_depth':    [10, 15, 20],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt', 0.8]
}

rf_base = RandomForestRegressor(random_state=42, n_jobs=-1)
grid_search = GridSearchCV(rf_base, param_grid, cv=5, scoring='r2',
                           n_jobs=-1, verbose=0)
t0 = time.time()
grid_search.fit(X_train, y_train)
t = time.time() - t0

best_rf = grid_search.best_estimator_
pred_rf = best_rf.predict(X_test)
results.append(evaluate('Random Forest (tuned)', y_test, pred_rf, t))

print("Best params:", grid_search.best_params_)
print(f"Random Forest (tuned) | R²={results[-1]['R2']:.4f} | MAE={results[-1]['MAE']:.3f}")

## 3. Gradient Boosting — Tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

gb_params = {
    'n_estimators':    [100, 150, 200],
    'learning_rate':   [0.05, 0.1, 0.15],
    'max_depth':       [3, 5, 7],
    'subsample':       [0.8, 1.0],
    'min_samples_leaf':[1, 3]
}

gb_base = GradientBoostingRegressor(random_state=42)
rand_search = RandomizedSearchCV(gb_base, gb_params, n_iter=20,
                                  cv=5, scoring='r2', n_jobs=-1,
                                  random_state=42, verbose=0)
t0 = time.time()
rand_search.fit(X_train, y_train)
t = time.time() - t0

best_gb = rand_search.best_estimator_
pred_gb = best_gb.predict(X_test)
results.append(evaluate('Gradient Boosting (tuned)', y_test, pred_gb, t))

print("Best params:", rand_search.best_params_)
print(f"Gradient Boosting | R²={results[-1]['R2']:.4f} | MAE={results[-1]['MAE']:.3f}")

## 4. Model Comparison

In [ ]:
df_results = pd.DataFrame(results).sort_values('R2', ascending=False)
print(df_results.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#2ecc71' if r == df_results['R2'].max() else '#95a5a6' for r in df_results['R2']]

df_results.plot(x='Model', y='R2', kind='bar', ax=axes[0], color=colors, legend=False, edgecolor='white')
axes[0].set_title('R² Score (higher = better)'); axes[0].set_xlabel(''); axes[0].set_ylim(0.5, 1.0)
axes[0].tick_params(axis='x', rotation=35)

df_results.plot(x='Model', y='MAE', kind='bar', ax=axes[1], color=colors, legend=False, edgecolor='white')
axes[1].set_title('MAE (lower = better)'); axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=35)

df_results.plot(x='Model', y='RMSE', kind='bar', ax=axes[2], color=colors, legend=False, edgecolor='white')
axes[2].set_title('RMSE (lower = better)'); axes[2].set_xlabel('')
axes[2].tick_params(axis='x', rotation=35)

plt.suptitle('Model Comparison — All Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/08_model_comparison.png', bbox_inches='tight')
plt.show()

## 5. Best Model — Detailed Analysis

In [ ]:
# Best model = Random Forest (tuned)
best_model = best_rf
best_pred  = pred_rf

# Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, best_pred, alpha=0.5, color='steelblue', s=30, edgecolors='none')
lo, hi = min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())
axes[0].plot([lo, hi], [lo, hi], 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Energy (kWh)'); axes[0].set_ylabel('Predicted Energy (kWh)')
axes[0].set_title('Actual vs Predicted — Random Forest')
axes[0].legend()

residuals = y_test - best_pred
axes[1].scatter(best_pred, residuals, alpha=0.5, color='coral', s=30, edgecolors='none')
axes[1].axhline(0, color='black', linewidth=1.2, linestyle='--')
axes[1].set_xlabel('Predicted Energy (kWh)'); axes[1].set_ylabel('Residual (Actual - Predicted)')
axes[1].set_title('Residual Plot')

plt.tight_layout()
plt.savefig('../outputs/09_actual_vs_predicted.png', bbox_inches='tight')
plt.show()

print(f"R²   : {r2_score(y_test, best_pred):.4f}")
print(f"MAE  : {mean_absolute_error(y_test, best_pred):.3f} kWh")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, best_pred)):.3f} kWh")

## 6. Feature Importance

In [ ]:
importances = pd.Series(best_model.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 10))
colors_fi = ['#2ecc71' if v >= importances.quantile(0.75) else '#bdc3c7' for v in importances]
importances.plot(kind='barh', ax=ax, color=colors_fi, edgecolor='white')
ax.set_title('Feature Importance — Random Forest (tuned)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.axvline(importances.mean(), color='red', linestyle='--', linewidth=1, label=f'Mean: {importances.mean():.3f}')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/10_feature_importance.png', bbox_inches='tight')
plt.show()

print("\nTop 10 features:")
print(importances.sort_values(ascending=False).head(10).round(4))

## 7. Cross-Validation Score

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='r2', n_jobs=-1)
print(f"5-Fold CV R² scores : {cv_scores.round(4)}")
print(f"Mean CV R²          : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

## 8. Save Best Model

In [ ]:
joblib.dump(best_model, '../models/best_model.pkl')
print("Best model saved to ../models/best_model.pkl")

# Save results table
df_results.to_csv('../outputs/model_comparison_results.csv', index=False)
print("Model comparison results saved.")

print("\n=== FINAL RESULTS ===")
print(df_results[['Model','R2','MAE','RMSE']].to_string(index=False))